# chromadose Quick Start

This notebook demonstrates the full chromadose workflow:
1. Build calibration from known dose-pixel pairs
2. Convert a film scan to a dose map using multiple methods
3. Compare methods and visualize results
4. Run gamma analysis
5. Generate a PDF QA report

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import chromadose as cd
from chromadose.core.types import FitParams, FilmScan
from chromadose.methods import MickeSolver, MayerSolver
from chromadose.analysis import gamma_2d, extract_row_profile, compare_profiles

print(f"chromadose v{cd.__version__}")

## 1. Create Calibration

In practice, you'd load real TIFF scans of calibration film strips.
Here we use synthetic data with realistic EBT3 parameters.

In [ ]:
# Known calibration doses (Gy)
doses = np.array([0.0, 0.5, 1.0, 2.0, 4.0, 7.0, 9.0])

# Realistic EBT3 rational function parameters:
# pixel(D) = (r + s*D) / (t + D)
KNOWN_RED = FitParams(r=0.655, s=0.037, t=2.956)
KNOWN_GREEN = FitParams(r=0.448, s=0.070, t=10.636)
KNOWN_BLUE = FitParams(r=0.402, s=0.007, t=5.963)

# Simulate calibration pixel values
red_pixels = KNOWN_RED.pixel(doses)
green_pixels = KNOWN_GREEN.pixel(doses)
blue_pixels = KNOWN_BLUE.pixel(doses)

# Build calibration
cal = cd.Calibration.from_arrays(
    doses=doses,
    red_pixels=red_pixels,
    green_pixels=green_pixels,
    blue_pixels=blue_pixels,
)
print(cal.summary())

In [ ]:
# Plot calibration curves
cal.plot_curves()
plt.tight_layout()
plt.show()

## 2. Create a Synthetic Treatment Film

We'll make a 200x200 film with a 2D dose pattern resembling
an IMRT field — a central high-dose region with penumbra falloff.

In [ ]:
# Create a 2D dose pattern: Gaussian beam profile
y, x = np.mgrid[0:200, 0:200].astype(np.float64)
true_dose = 4.0 * np.exp(-((x - 100)**2 + (y - 100)**2) / (2 * 40**2))
true_dose += 1.0 * np.exp(-((x - 60)**2 + (y - 140)**2) / (2 * 20**2))

# Generate film pixel values from the dose pattern
rng = np.random.default_rng(42)
noise = 0.003  # realistic scanner noise

film = FilmScan(
    red=KNOWN_RED.pixel(true_dose) + rng.normal(0, noise, true_dose.shape),
    green=KNOWN_GREEN.pixel(true_dose) + rng.normal(0, noise, true_dose.shape),
    blue=KNOWN_BLUE.pixel(true_dose) + rng.normal(0, noise, true_dose.shape),
    dpi=72.0,
)

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
axes[0].imshow(true_dose, cmap='jet'); axes[0].set_title('True Dose (Gy)')
axes[1].imshow(film.red, cmap='Reds_r'); axes[1].set_title('Red Channel')
axes[2].imshow(film.green, cmap='Greens_r'); axes[2].set_title('Green Channel')
axes[3].imshow(film.blue, cmap='Blues_r'); axes[3].set_title('Blue Channel')
for ax in axes: ax.axis('off')
plt.colorbar(axes[0].images[0], ax=axes[0], fraction=0.046)
plt.tight_layout()
plt.show()

## 3. Solve for Dose — Multiple Methods

In [ ]:
import time

# Micke method (original multichannel)
t0 = time.time()
micke_result = MickeSolver().solve(film, cal.result)
print(f"Micke:  {time.time()-t0:.3f}s  max={micke_result.dose.max():.2f} Gy")

# Mayer method (analytical)
t0 = time.time()
mayer_result = MayerSolver().solve(film, cal.result)
print(f"Mayer:  {time.time()-t0:.3f}s  max={mayer_result.dose.max():.2f} Gy")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

vmax = true_dose.max()
im0 = axes[0].imshow(micke_result.dose, cmap='jet', vmin=0, vmax=vmax)
axes[0].set_title(f'Micke Dose (max={micke_result.dose.max():.2f} Gy)')

im1 = axes[1].imshow(mayer_result.dose, cmap='jet', vmin=0, vmax=vmax)
axes[1].set_title(f'Mayer Dose (max={mayer_result.dose.max():.2f} Gy)')

diff = micke_result.dose - mayer_result.dose
im2 = axes[2].imshow(diff, cmap='RdBu_r', vmin=-0.1, vmax=0.1)
axes[2].set_title(f'Micke - Mayer (max diff={np.abs(diff).max():.3f} Gy)')

for ax, im in zip(axes, [im0, im1, im2]):
    plt.colorbar(im, ax=ax, fraction=0.046)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 4. Gamma Analysis

Compare the film measurement against the "true" dose (simulating TPS comparison).

In [ ]:
gamma_result = gamma_2d(
    reference=true_dose,
    evaluated=micke_result.dose,
    dose_criteria=3.0,
    distance_criteria_mm=3.0,
    pixel_size_mm=film.pixel_size_mm,
    dose_threshold_pct=10.0,
)

print(f"Gamma {gamma_result.criteria}")
print(f"  Pass rate: {gamma_result.pass_rate * 100:.1f}%")
print(f"  Points evaluated: {gamma_result.points_evaluated}")
print(f"  Points passed: {gamma_result.points_passed}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

im0 = axes[0].imshow(gamma_result.gamma_map, cmap='RdYlGn_r', vmin=0, vmax=2)
axes[0].set_title('Gamma Index Map')
plt.colorbar(im0, ax=axes[0], fraction=0.046)

# Pass/fail
gmap = gamma_result.gamma_map
pass_fail = np.where(np.isnan(gmap), 0.5, np.where(gmap <= 1.0, 1.0, 0.0))
axes[1].imshow(pass_fail, cmap='RdYlGn', vmin=0, vmax=1)
axes[1].set_title(f'Pass/Fail — {gamma_result.pass_rate*100:.1f}% pass rate')

for ax in axes: ax.axis('off')
plt.tight_layout()
plt.show()

## 5. Dose Profiles

In [ ]:
# Extract profiles through the center
ref_profile = extract_row_profile(true_dose, row=100, pixel_size_mm=film.pixel_size_mm, label='TPS')
film_profile = extract_row_profile(micke_result.dose, row=100, pixel_size_mm=film.pixel_size_mm, label='Film')

comparison = compare_profiles(ref_profile, film_profile)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), gridspec_kw={'height_ratios': [3, 1]})

ax1.plot(ref_profile.position_mm, ref_profile.dose, 'b-', lw=2, label='TPS (true)')
ax1.plot(film_profile.position_mm, film_profile.dose, 'r--', lw=2, label='Film (Micke)')
ax1.set_ylabel('Dose (Gy)')
ax1.legend()
ax1.set_title(f'Crossline Profile at y=100 — Mean diff: {comparison.mean_abs_diff_pct:.1f}%')
ax1.grid(True, alpha=0.3)

ax2.plot(ref_profile.position_mm, comparison.dose_diff_pct, 'k-', lw=1)
ax2.axhline(0, color='gray', ls='--')
ax2.set_ylabel('Diff (%)')
ax2.set_xlabel('Position (mm)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Generate PDF Report

In [ ]:
from chromadose.io import generate_report

report_path = generate_report(
    'qa_report.pdf',
    micke_result,
    gamma_result=gamma_result,
    profiles=[comparison],
    reference_dose=true_dose,
    title='IMRT QA Report — Head & Neck',
    patient_id='DEMO-001',
    plan_name='HN VMAT 6MV',
    analyst='chromadose notebook',
)
print(f'Report saved to {report_path}')

## Summary

chromadose provides a complete film dosimetry pipeline:

| Step | Module | Key function |
|---|---|---|
| Calibration | `chromadose.Calibration` | `from_arrays()`, `save()`, `load()` |
| Dose solving | `chromadose.methods` | `MickeSolver`, `MayerSolver`, `MultigaussianSolver`, `ANNSolver` |
| Gamma analysis | `chromadose.analysis` | `gamma_2d()` |
| Profiles | `chromadose.analysis` | `extract_row_profile()`, `compare_profiles()` |
| Registration | `chromadose.analysis` | `register_auto()`, `register_manual()` |
| DICOM import | `chromadose.io.dicom` | `load_dicom_dose()`, `resample_to_film()` |
| PDF reports | `chromadose.io` | `generate_report()` |
| CLI | `chromadose` | `calibrate`, `solve`, `gamma`, `report` |